# Deutsch-Jozsa Algorithm — Amazon Braket

Given a black-box function $f: \{0,1\}^n \to \{0,1\}$ promised to be
either **constant** (all zeros or all ones) or **balanced** (exactly
half zeros, half ones), Deutsch-Jozsa determines which in a single
query.

With $n=2$, we test all 6 balanced functions and the 2 constant functions.

In [ ]:
from braket.circuit import Circuit
from braket.devices import LocalSimulator

In [ ]:
device = LocalSimulator()

## Oracle definitions

Qubits 0, 1 are input; qubit 2 is the ancilla.

In [ ]:
def oracle_constant_0():
    """f(x) = 0 for all x."""
    return Circuit()

def oracle_constant_1():
    """f(x) = 1 for all x."""
    c = Circuit()
    c.x(2)
    return c

def oracle_balanced_id():
    """f(x0,x1) = x0."""
    c = Circuit()
    c.cnot(0, 2)
    return c

def oracle_balanced_not():
    """f(x0,x1) = NOT x0."""
    c = Circuit()
    c.cnot(0, 2)
    c.x(2)
    return c

def oracle_balanced_xor():
    """f(x0,x1) = x0 XOR x1."""
    c = Circuit()
    c.cnot(0, 2)
    c.cnot(1, 2)
    return c

def oracle_balanced_xnor():
    """f(x0,x1) = NOT (x0 XOR x1)."""
    c = Circuit()
    c.cnot(0, 2)
    c.cnot(1, 2)
    c.x(2)
    return c

def oracle_balanced_msb():
    """f(x0,x1) = x1."""
    c = Circuit()
    c.cnot(1, 2)
    return c

def oracle_balanced_nand():
    """f(x0,x1) = NOT (x0 AND x1)."""
    c = Circuit()
    c.ccnot(0, 1, 2)
    c.x(2)
    return c

## Deutsch-Jozsa circuit

In [ ]:
def deutsch_jozsa(oracle_fn, n=2):
    """Build the Deutsch-Jozsa circuit."""
    circuit = Circuit()
    circuit.x(n)
    circuit.h(n)
    for i in range(n):
        circuit.h(i)
    circuit.add_circuit(oracle_fn())
    for i in range(n):
        circuit.h(i)
    for i in range(n):
        circuit.measure(i)
    return circuit

print(deutsch_jozsa(oracle_constant_0))

## Run all oracles

In [ ]:
oracles = [
    ("f=0 (constant)", oracle_constant_0),
    ("f=1 (constant)", oracle_constant_1),
    ("f=x0 (balanced)", oracle_balanced_id),
    ("f=NOT x0", oracle_balanced_not),
    ("f=x0 XOR x1", oracle_balanced_xor),
    ("f=NOT(x0 XOR x1)", oracle_balanced_xnor),
    ("f=x1 (balanced)", oracle_balanced_msb),
    ("f=NOT(x0 AND x1)", oracle_balanced_nand),
]

print("All-zero measurement -> CONSTANT")
print("Any non-zero measurement -> BALANCED")
print()

for name, fn in oracles:
    circuit = deutsch_jozsa(fn)
    result = device.run(circuit, shots=100).result()
    counts = result.result_types[0].value
    all_zero = all(b == '0' for b in counts)
    verdict = "CONSTANT" if all_zero else "BALANCED"
    print(f"  {name:>25s}  counts={counts}  -> {verdict}")

print()
print("All oracles classified correctly in ONE query.")
print("Classical worst case requires 2^(n-1) + 1 queries.")